# PillSight — Model 3: ViT-B/16 (v2)
**ADSP 31018 | University of Chicago**

> Set Runtime → T4/A100 GPU before running!

## 0. GPU check

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

## 1. Clone repo & install deps

In [ ]:
import os
REPO_URL  = 'https://github.com/Devanshu1503/ML2_Class_Project'
REPO_NAME = 'ML2_Class_Project'
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    print('Pulling latest...')
    !cd {REPO_NAME} && git pull
os.chdir(f'/content/{REPO_NAME}')
print('CWD:', os.getcwd())

In [ ]:
!pip install -q scikit-learn
print('✓ deps ready')

## 2. Download dataset

In [ ]:
import zipfile, pathlib, os
DATA_URL = 'https://github.com/usuyama/ePillID-benchmark/releases/download/ePillID_data_v1.0/ePillID_data.zip'
ZIP_PATH = '/content/ePillID_data.zip'
DATA_DIR = '/content/ML2_Class_Project/data'
if not pathlib.Path(DATA_DIR).exists():
    print('Downloading (~153 MB)...')
    !wget -q --show-progress -O {ZIP_PATH} {DATA_URL}
    print('Unzipping...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/ML2_Class_Project/')
    os.rename('/content/ML2_Class_Project/ePillID_data', DATA_DIR)
    print('✓ Dataset ready')
else:
    print('✓ Already present')
!ls /content/ML2_Class_Project/data/

## 3. Set env var

In [ ]:
import os
os.environ['COLAB_ROOT'] = '/content/ML2_Class_Project'
print('COLAB_ROOT:', os.environ['COLAB_ROOT'])

## 4. Run data_exploration.py

In [ ]:
import pathlib
if pathlib.Path('data/filtered_metadata.csv').exists():
    print('✓ filtered_metadata.csv exists — skipping')
else:
    !cd model1_cnn && python data_exploration.py

## 5. 🔍 DEBUG: Inspect data before training

This cell is critical — it tells us if the data pipeline is correct.

In [ ]:
import sys, os, json
import pandas as pd
from PIL import Image

sys.path.insert(0, '/content/ML2_Class_Project/model3_vit')
os.environ['COLAB_ROOT'] = '/content/ML2_Class_Project'

# Load CSV and class map
df = pd.read_csv('/content/ML2_Class_Project/data/filtered_metadata.csv')
with open('/content/ML2_Class_Project/data/class_map.json') as f:
    class_map = json.load(f)

print('=== DATA AUDIT ===')
print(f'Total rows:    {len(df)}')
print(f'Columns:       {list(df.columns)}')
print(f'Num classes:   {len(class_map)}')
print(f'Class map:     {class_map}')
print(f'\nLabel counts:')
print(df['label_code_id'].value_counts())

# Check image paths
IMAGE_ROOT = '/content/ML2_Class_Project/data/classification_data'
print(f'\n=== IMAGE PATH CHECK ===')
print(f'IMAGE_ROOT: {IMAGE_ROOT}')
print(f'Contents of IMAGE_ROOT:')
for f in os.listdir(IMAGE_ROOT)[:5]:
    print(' ', f)

# Try to open first 5 images
print('\nFirst 5 image paths in CSV:')
for i, row in df.head(5).iterrows():
    full_path = os.path.join(IMAGE_ROOT, row['image_path'])
    exists = os.path.exists(full_path)
    print(f'  {"✓" if exists else "✗"} {full_path}')
    if exists:
        img = Image.open(full_path)
        print(f'      size={img.size}, mode={img.mode}')

# Check labels are sane integers
print('\n=== LABEL SANITY ===')
labels_in_df  = set(str(x) for x in df['label_code_id'].unique())
labels_in_map = set(class_map.keys())
print(f'Labels in CSV not in class_map: {labels_in_df - labels_in_map}')
print(f'Labels in class_map not in CSV: {labels_in_map - labels_in_df}')
if labels_in_df == labels_in_map:
    print('✓ Labels match perfectly!')
else:
    print('✗ MISMATCH — this is likely the training bug!')

## 6. Train ViT-B/16

In [ ]:
# Clean old checkpoint so we start fresh
import os
ckpt = '/content/ML2_Class_Project/model3_vit/checkpoints/best_model.pth'
if os.path.exists(ckpt):
    os.remove(ckpt)
    print('Removed old checkpoint')
os.chdir('/content/ML2_Class_Project/model3_vit')
!python train.py

## 7. Evaluate

In [ ]:
os.chdir('/content/ML2_Class_Project/model3_vit')
!python evaluate.py

## 8. Attention maps

In [ ]:
os.chdir('/content/ML2_Class_Project/model3_vit')
!python attention_map.py

## 9. Display results

In [ ]:
import json, glob, os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

BASE = '/content/ML2_Class_Project/model3_vit'

lc = f'{BASE}/outputs/learning_curves.png'
if os.path.exists(lc):
    plt.figure(figsize=(10,4)); plt.imshow(mpimg.imread(lc)); plt.axis('off'); plt.show()

m = f'{BASE}/outputs/metrics.json'
if os.path.exists(m):
    metrics = json.load(open(m))
    print('\n── Test metrics ──────────────────')
    for k,v in metrics.items(): print(f'  {k:20s}: {v:.4f}')

attn_imgs = sorted(glob.glob(f'{BASE}/outputs/attention_maps/*.png'))[:3]
if attn_imgs:
    fig, axes = plt.subplots(1, len(attn_imgs), figsize=(14,5))
    if len(attn_imgs)==1: axes=[axes]
    for ax, p in zip(axes, attn_imgs):
        ax.imshow(mpimg.imread(p)); ax.axis('off'); ax.set_title(os.path.basename(p), fontsize=7)
    plt.suptitle('Attention Maps'); plt.tight_layout(); plt.show()

## 10. Save to Google Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree('/content/ML2_Class_Project/model3_vit/checkpoints',
#                 '/content/drive/MyDrive/ML2_PillSight/checkpoints', dirs_exist_ok=True)
# shutil.copytree('/content/ML2_Class_Project/model3_vit/outputs',
#                 '/content/drive/MyDrive/ML2_PillSight/outputs', dirs_exist_ok=True)
# print('✓ Saved to Drive')
print('Uncomment above to save to Drive.')